# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

## Step 1: Create Spark session

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

## Step 2: Load the data
Load the citation data and patent data as Spark DataFrames, using 
inferSchema to automatically detect column types.


In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Step 3: Build a patent-to-state lookup
Extract just the patent number and state (POSTATE) from the patent data. 
Used twice: once for the citing patent's state, once for the cited patent's.

In [8]:
from pyspark.sql import functions as F

patent_state = patents.select(
    F.col("PATENT").alias("PATENT_ID"),
    F.col("POSTATE").alias("STATE")
)

## Step 4: Join citations to the citing patent's state

In [9]:
citing_joined = citations.join(
    patent_state.withColumnRenamed("PATENT_ID", "CITING").withColumnRenamed("STATE", "CITING_STATE"),
    on="CITING", how="left"
)
citing_joined.show(5)

+-------+-------+------------+
| CITING|  CITED|CITING_STATE|
+-------+-------+------------+
|3858242|1515701|          MI|
|3858241| 956203|          MA|
|3858241|1324234|          MA|
|3858241|3398406|          MA|
|3858241|3557384|          MA|
+-------+-------+------------+
only showing top 5 rows



## Step 5: Join to the cited patent's state
The result is the intermediate table described in the assignment: for 
every citation, we know both patents' states. Cached since it's reused below.

In [10]:
both_joined = citing_joined.join(
    patent_state.withColumnRenamed("PATENT_ID", "CITED").withColumnRenamed("STATE", "CITED_STATE"),
    on="CITED", how="left"
)

both_joined = both_joined.cache()
both_joined.show(10)

+-----+-------+------------+-----------+
|CITED| CITING|CITING_STATE|CITED_STATE|
+-----+-------+------------+-----------+
| 2366|4192521|        NULL|       NULL|
| 2366|4305315|          MN|       NULL|
| 2366|4253355|          MN|       NULL|
| 5156|5580635|          WI|       NULL|
| 5518|4976561|        NULL|       NULL|
| 5803|4480374|          MN|       NULL|
| 6620|5123817|        NULL|       NULL|
| 7240|4115020|        NULL|       NULL|
| 7253|4727698|          CA|       NULL|
| 7340|4108250|          IL|       NULL|
+-----+-------+------------+-----------+
only showing top 10 rows



## Step 6: Filter to same-state citations and count
Keep only citations where both states are known and equal. Group by 
citing patent and count.

In [11]:
same_state = both_joined.filter(
    F.col("CITING_STATE").isNotNull() &
    F.col("CITED_STATE").isNotNull() &
    (F.col("CITING_STATE") == F.col("CITED_STATE"))
)

same_state_counts = same_state.groupBy("CITING").agg(
    F.count("*").alias("SAME_STATE_CITATIONS")
).withColumnRenamed("CITING", "PATENT")

same_state_counts.show(5)

+-------+--------------------+
| PATENT|SAME_STATE_CITATIONS|
+-------+--------------------+
|5300411|                   7|
|3956677|                   1|
|4171110|                   4|
|4031936|                   2|
|3894399|                   3|
+-------+--------------------+
only showing top 5 rows



## Step 7: Augment patent data and find the top 10
Left-join the same-state counts onto the full patent table, filling 
missing values with 0. Sort descending for the top 10 self-cited patents.

In [12]:
augmented = patents.join(same_state_counts, on="PATENT", how="left") \
    .fillna(0, subset=["SAME_STATE_CITATIONS"])

top10 = augmented.orderBy(F.col("SAME_STATE_CITATIONS").desc()).limit(10)
top10.select("PATENT", "POSTATE", "SAME_STATE_CITATIONS").show()


+-------+-------+--------------------+
| PATENT|POSTATE|SAME_STATE_CITATIONS|
+-------+-------+--------------------+
|5959466|     CA|                 125|
|5983822|     TX|                 103|
|6008204|     CA|                 100|
|5952345|     CA|                  98|
|5958954|     CA|                  96|
|5998655|     CA|                  96|
|5936426|     CA|                  94|
|5739256|     CA|                  90|
|5913855|     CA|                  90|
|5925042|     CA|                  90|
+-------+-------+--------------------+

